# Social and Demographic Features

**Purpose.** ACS DP02 household composition, education, mobility, language, disability, veteran, and technology-access features.

This notebook is a read-only diagnostic consumer of the current `feature.*` layer. All transformations and canonical ACS mappings are owned by `src/cli/build_database.py`.

## Setup and current feature definitions

In [ ]:
from pathlib import Path
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 150)
sns.set_theme(style="whitegrid")

ROOT = Path.cwd()
while not (ROOT / "data" / "housing_predict.duckdb").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DB_PATH = ROOT / "data" / "housing_predict.duckdb"
if not DB_PATH.exists():
    raise FileNotFoundError("Run `python -m src.cli.build_database` from the project root first.")

con = duckdb.connect(str(DB_PATH), read_only=True)
FEATURE_TABLES = ['county_social_annual']

available = {row[0] for row in con.execute(
    "SELECT table_name FROM information_schema.tables WHERE table_schema = 'feature'"
).fetchall()}
missing = sorted(set(FEATURE_TABLES) - available)
if missing:
    raise RuntimeError(
        f"Database is missing current feature tables {missing}. "
        "Run `python -m src.cli.build_database` to rebuild it."
    )

registry_available = "canonical_feature_registry" in available
if registry_available:
    placeholders = ", ".join("?" for _ in FEATURE_TABLES)
    source_tables = [{
        "county_economic_annual": "dp03",
        "county_social_annual": "dp02",
        "county_housing": "dp04",
    }.get(table) for table in FEATURE_TABLES]
    source_tables = [table for table in source_tables if table]
    if source_tables:
        source_placeholders = ", ".join("?" for _ in source_tables)
        registry = con.execute(
            f"SELECT feature_name, acs_table, unit, min(year) AS first_year, "
            f"max(year) AS last_year, count(*) AS mapped_years "
            f"FROM feature.canonical_feature_registry WHERE acs_table IN ({source_placeholders}) "
            f"GROUP BY 1,2,3 ORDER BY 2,1",
            source_tables,
        ).df()
    else:
        registry = pd.DataFrame(columns=["feature_name", "acs_table", "unit", "first_year", "last_year", "mapped_years"])
else:
    registry = pd.DataFrame()
display(registry)

## Table inventory and county-year grain

In [ ]:
inventory = []
for table in FEATURE_TABLES:
    schema = con.execute(f"DESCRIBE feature.{table}").df()
    row_count, distinct_keys, min_year, max_year = con.execute(
        f"SELECT count(*), count(DISTINCT (county_fips, year)), min(year), max(year) FROM feature.{table}"
    ).fetchone()
    inventory.append({
        "feature_table": table,
        "rows": row_count,
        "distinct_county_years": distinct_keys,
        "duplicate_county_years": row_count - distinct_keys,
        "first_year": min_year,
        "last_year": max_year,
        "feature_columns": len(schema) - len({"county_fips", "year", "county_name"} & set(schema["column_name"])),
    })
inventory = pd.DataFrame(inventory)
display(inventory)
assert (inventory["duplicate_county_years"] == 0).all(), "Feature tables must be unique by county and year."

## Completeness by feature and year

In [ ]:
completeness = []
identifier_columns = {"county_fips", "year", "county_name"}
for table in FEATURE_TABLES:
    columns = con.execute(f"DESCRIBE feature.{table}").df()["column_name"].tolist()
    for column in [column for column in columns if column not in identifier_columns]:
        quoted = '"' + column.replace('"', '""') + '"'
        rows = con.execute(
            f"SELECT year, count(*) AS rows, count({quoted}) AS nonnull_rows "
            f"FROM feature.{table} GROUP BY year ORDER BY year"
        ).df()
        rows["feature_table"] = table
        rows["feature_name"] = column
        rows["completeness_pct"] = 100 * rows["nonnull_rows"] / rows["rows"]
        completeness.append(rows)
completeness = pd.concat(completeness, ignore_index=True)
display(completeness.groupby(["feature_table", "feature_name"], as_index=False).agg(
    first_available_year=("year", lambda s: completeness.loc[s.index][completeness.loc[s.index, "nonnull_rows"] > 0]["year"].min()),
    mean_completeness_pct=("completeness_pct", "mean"),
    minimum_completeness_pct=("completeness_pct", "min"),
).sort_values(["feature_table", "mean_completeness_pct"]))

## Distributions and range checks

In [ ]:
summaries = []
for table in FEATURE_TABLES:
    schema = con.execute(f"DESCRIBE feature.{table}").df()
    numeric = schema.loc[
        schema["column_type"].str.contains("DOUBLE|FLOAT|DECIMAL|INTEGER|BIGINT", case=False, regex=True),
        "column_name",
    ].tolist()
    for column in [column for column in numeric if column != "year"]:
        q = '"' + column.replace('"', '""') + '"'
        values = con.execute(f"SELECT {q} FROM feature.{table} WHERE {q} IS NOT NULL").df().iloc[:, 0]
        if values.empty:
            continue
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        summaries.append({
            "feature_table": table, "feature_name": column, "count": len(values),
            "min": values.min(), "median": values.median(), "mean": values.mean(), "max": values.max(),
            "iqr_outliers": int(((values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)).sum()),
            "outside_percent_range": int(((values < 0) | (values > 100)).sum()) if column.endswith("_pct") else np.nan,
        })
summaries = pd.DataFrame(summaries)
display(summaries)
assert summaries["outside_percent_range"].fillna(0).eq(0).all(), "Percentage feature outside [0, 100]."

## Correlation and redundancy

In [ ]:
for table in FEATURE_TABLES:
    frame = con.execute(f"SELECT * FROM feature.{table}").df()
    numeric = frame.select_dtypes(include=np.number).drop(columns=["year"], errors="ignore")
    if numeric.shape[1] < 2:
        continue
    corr = numeric.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    fig, ax = plt.subplots(figsize=(max(8, 0.45 * len(corr)), max(6, 0.4 * len(corr))))
    sns.heatmap(corr, mask=mask, cmap="vlag", center=0, vmin=-1, vmax=1, ax=ax)
    ax.set_title(f"{table}: feature correlations")
    plt.tight_layout()
    plt.show()

    pairs = corr.where(~mask).stack().rename("correlation").reset_index()
    pairs.columns = ["feature_a", "feature_b", "correlation"]
    display(pairs.assign(abs_correlation=pairs["correlation"].abs()).sort_values("abs_correlation", ascending=False).head(20))

## Temporal coverage and trends

In [ ]:
for table in FEATURE_TABLES:
    schema = con.execute(f"DESCRIBE feature.{table}").df()
    numeric = schema.loc[schema["column_type"].str.contains("DOUBLE|FLOAT|DECIMAL|INTEGER|BIGINT", case=False), "column_name"].tolist()
    numeric = [column for column in numeric if column != "year"]
    if not numeric:
        continue
    expressions = ", ".join(f'avg("{column}") AS "{column}"' for column in numeric)
    annual = con.execute(f"SELECT year, {expressions} FROM feature.{table} GROUP BY year ORDER BY year").df()
    normalized = annual.set_index("year").apply(lambda s: (s - s.mean()) / s.std() if s.std() else 0)
    fig, ax = plt.subplots(figsize=(11, 5))
    normalized.plot(ax=ax, legend=False, alpha=0.7)
    ax.set_title(f"{table}: standardized annual county means")
    ax.set_ylabel("Standard deviations from feature mean")
    plt.tight_layout()
    plt.show()
    display(annual)

con.close()

## Interpretation notes

- ACS values are five-year estimates associated with their release year. Adjacent years therefore contain overlapping survey periods.
- Monetary ACS features and the housing target are expressed in 2024 dollars.
- A null before a feature's `first_year` can indicate that ACS had not yet published a reconciled definition.
- `target_median_owner_occupied_home_value_2024_usd` is the official DP04 median, because DP04 does not publish a county average housing-unit value.
- The canonical registry and transactional build are the authority for cross-year compatibility.